1 - Install Library

In [1]:
!pip install -q sentence-transformers faiss-cpu transformers accelerate pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 46.4 MB/s eta 0:00:00


2 - Import Library

In [2]:
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer
import faiss

from transformers import pipeline

3 - Upload Dataset

In [7]:
from google.colab import files

uploaded = files.upload()

Saving movies.csv.zip to movies.csv.zip
Saving ratings.csv.zip to ratings.csv.zip


4 - Load Dataset

In [10]:
import zipfile

with zipfile.ZipFile("ratings.csv.zip", 'r') as zip_ref:
    zip_ref.extractall()

with zipfile.ZipFile("movies.csv.zip", 'r') as zip_ref:
    zip_ref.extractall()

ratings = pd.read_csv("ratings.csv")
movies = pd.read_csv("movies.csv")

print(ratings.head())
print(movies.head())

   userId  movieId  rating   timestamp
0       1      296     5.0  1147880044
1       1      306     3.5  1147868817
2       1      307     5.0  1147868828
3       1      665     5.0  1147878820
4       1      899     3.5  1147868510
   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  


5 - Preprocessing
Hitung rata-rata rating setiap film.

In [11]:
movie_stats = ratings.groupby("movieId").agg(
    avg_rating=("rating","mean"),
    total_rating=("rating","count")
).reset_index()

movies = movies.merge(
    movie_stats,
    on="movieId",
    how="left"
)

movies.fillna({
    "avg_rating":0,
    "total_rating":0
}, inplace=True)

movies.head()

,movieId,title,genres,avg_rating,total_rating
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,3.893708,57309.0
1,2,Jumanji (1995),Adventure|Children|Fantasy,3.251527,24228.0
2,3,Grumpier Old Men (1995),Comedy|Romance,3.142028,11804.0
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,2.853547,2523.0
4,5,Father of the Bride Part II (1995),Comedy,3.058434,11714.0


6 - Membuat Dokumen untuk RAG

In [12]:
documents = []

for _, row in movies.iterrows():

    doc = f"""
Movie Title : {row['title']}

Genres : {row['genres']}

Average Rating : {row['avg_rating']:.2f}

Total Ratings : {int(row['total_rating'])}
"""

    documents.append(doc)

print(documents[0])


Movie Title : Toy Story (1995)

Genres : Adventure|Animation|Children|Comedy|Fantasy

Average Rating : 3.89

Total Ratings : 57309



7 - Embedding

In [13]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

embeddings = embedding_model.encode(
    documents,
    show_progress_bar=True,
    convert_to_numpy=True
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1951 [00:00<?, ?it/s]

8 - Membuat FAISS Index

In [14]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

print("Jumlah film :", index.ntotal)

Jumlah film : 62423


9 - Load LLM

Menggunakan Flan-T5.

In [16]:
generator = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    max_new_tokens=256
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

...

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', '

10 - Retrieval Function

In [17]:
def retrieve_movies(query, top_k=5):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    )

    distances, indices = index.search(
        query_embedding,
        top_k
    )

    results = movies.iloc[indices[0]]

    return results

11 - RAG Function

In [18]:
def recommend(query):

    retrieved = retrieve_movies(query)

    context = ""

    for _, row in retrieved.iterrows():

        context += f"""
Title : {row['title']}
Genres : {row['genres']}
Average Rating : {row['avg_rating']:.2f}
Total Ratings : {int(row['total_rating'])}

"""

    prompt = f"""
You are a movie recommendation assistant.

User Request:
{query}

Retrieved Movies:
{context}

Recommend the best movies with explanations.
"""

    answer = generator(prompt)[0]["generated_text"]

    return retrieved, answer

12 - Testing

In [19]:
movies_found, answer = recommend(
    "I like science fiction movies with high ratings."
)

movies_found

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


,movieId,title,genres,avg_rating,total_rating
56918,195217,Star Wars: Dresca,Sci-Fi,2.833333,3.0
51256,182745,Alistair1918 (2016),Sci-Fi,2.500000,1.0
44069,167448,Domain (2016),Drama|Sci-Fi|Thriller,3.857143,7.0
49367,178731,The First (2014),Sci-Fi,2.500000,2.0
20915,108085,+1 (2013),Sci-Fi|Thriller,2.900000,55.0


13 - Hasil LLM

In [20]:
print(answer)


You are a movie recommendation assistant.

User Request:
I like science fiction movies with high ratings.

Retrieved Movies:

Title : Star Wars: Dresca
Genres : Sci-Fi
Average Rating : 2.83
Total Ratings : 3


Title : Alistair1918 (2016)
Genres : Sci-Fi
Average Rating : 2.50
Total Ratings : 1


Title : Domain (2016)
Genres : Drama|Sci-Fi|Thriller
Average Rating : 3.86
Total Ratings : 7


Title : The First (2014)
Genres : Sci-Fi
Average Rating : 2.50
Total Ratings : 2


Title : +1 (2013)
Genres : Sci-Fi|Thriller
Average Rating : 2.90
Total Ratings : 55



Recommend the best movies with explanations.



14 - Versi Interaktif

In [23]:
# Input preferensi pengguna
query = input("Masukkan preferensi film: ")

# Jalankan RAG Recommendation
retrieved, answer = recommend(query)

print("\n==========================")
print("Top Retrieved Movies")
print("==========================")

display(
    retrieved[
        ["title", "genres", "avg_rating"]
    ]
)

print("\n==========================")
print("LLM Recommendation")
print("==========================")

print(answer)

print("\nProgram selesai.")

Masukkan preferensi film: i like scary movies


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Top Retrieved Movies


,title,genres,avg_rating
20952,American Scary (2006),Comedy|Documentary|Horror,4.000000
10761,Scary Movie 4 (2006),Comedy|Horror,2.318713
12299,Are You Scared? (2006),Horror,1.839286
3684,Scary Movie (2000),Comedy|Horror,2.717772
60395,Scary Stories to Tell in the Dark (2019),Horror,3.159420



LLM Recommendation

You are a movie recommendation assistant.

User Request:
i like scary movies

Retrieved Movies:

Title : American Scary (2006)
Genres : Comedy|Documentary|Horror
Average Rating : 4.00
Total Ratings : 1


Title : Scary Movie 4 (2006)
Genres : Comedy|Horror
Average Rating : 2.32
Total Ratings : 1881


Title : Are You Scared? (2006)
Genres : Horror
Average Rating : 1.84
Total Ratings : 28


Title : Scary Movie (2000)
Genres : Comedy|Horror
Average Rating : 2.72
Total Ratings : 11473


Title : Scary Stories to Tell in the Dark (2019)
Genres : Horror
Average Rating : 3.16
Total Ratings : 69



Recommend the best movies with explanations.


Program selesai.


UAS PRAKTEK SISTEM REKOMENDASI

menyimpan model yang sudah ditraining di Google Colab

15 - Simpan Model RAG ke Google Drive

Mount Google Drive terlebih dahulu.

In [39]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Kemudian simpan seluruh komponen.

In [40]:
import os
import pickle
import faiss
import shutil

save_path = "/content/drive/MyDrive/RAG_Movie_API"

os.makedirs(save_path, exist_ok=True)

# 1. Simpan Index FAISS
faiss.write_index(index, f"{save_path}/movie_index.faiss")

# 2. Simpan Metadata Film
movies.to_csv(f"{save_path}/movies_metadata.csv", index=False)

# 3. Simpan daftar dokumen
with open(f"{save_path}/documents.pkl","wb") as f:
    pickle.dump(documents,f)

# 4. Simpan model embedding
embedding_model.save(f"{save_path}/embedding_model")

print("Semua file berhasil disimpan.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Semua file berhasil disimpan.


16 - Membuat API (api.py)

In [41]:
%%writefile api.py

from fastapi import FastAPI
import pandas as pd
import faiss
import pickle

from sentence_transformers import SentenceTransformer
from transformers import pipeline

app = FastAPI(title="Movie Recommendation API")

# =============================
# Load Model
# =============================

PATH = "/content/drive/MyDrive/RAG_Movie_API"

index = faiss.read_index(
    f"{PATH}/movie_index.faiss"
)

movies = pd.read_csv(
    f"{PATH}/movies_metadata.csv"
)

with open(f"{PATH}/documents.pkl","rb") as f:
    documents = pickle.load(f)

embedding_model = SentenceTransformer(
    f"{PATH}/embedding_model"
)

generator = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    max_new_tokens=256
)

# =============================
# Root
# =============================

@app.get("/")
def root():

    return {
        "message":"RAG Movie Recommendation API Aktif"
    }

# =============================
# Recommendation
# =============================

@app.get("/recommend")

def recommend(
    query:str,
    top_k:int=5
):

    query_embedding = embedding_model.encode(
        [query]
    )

    distances, indices = index.search(
        query_embedding,
        top_k
    )

    retrieved = movies.iloc[
        indices[0]
    ]

    context=""

    for _,row in retrieved.iterrows():

        context += f"""

Title : {row['title']}

Genres : {row['genres']}

Average Rating : {row['avg_rating']:.2f}

"""

    prompt=f"""
You are movie recommendation assistant.

User Request:
{query}

Retrieved Movies:

{context}

Recommend the best movies with explanations.
"""

    answer=generator(prompt)[0]["generated_text"]

    return {
        "query":query,
        "recommendations":
        retrieved[
            [
                "title",
                "genres",
                "avg_rating"
            ]
        ].to_dict(orient="records"),
        "llm_answer":answer
    }


Overwriting api.py


17 - Menjalankan API

In [46]:
!pip install fastapi uvicorn -q

In [47]:
import subprocess
import time

print("Menjalankan API...")

process = subprocess.Popen([
    "uvicorn",
    "api:app",
    "--host",
    "127.0.0.1",
    "--port",
    "8000"
])

time.sleep(5)

print("API aktif pada http://127.0.0.1:8000")

Menjalankan API...
API aktif pada http://127.0.0.1:8000


18 - Client CLI

In [48]:
import requests

API = "http://127.0.0.1:8000/recommend"

print("="*60)
print("RAG MOVIE RECOMMENDATION SYSTEM")
print("="*60)

query = input("Masukkan preferensi film : ")

response = requests.get(
    API,
    params={
        "query":query,
        "top_k":5
    }
)

if response.status_code==200:

    result=response.json()

    print("\nTop Retrieved Movies\n")

    for i,movie in enumerate(
        result["recommendations"],1
    ):

        print(
            f"{i}. {movie['title']}"
        )

    print("\nPenjelasan LLM\n")

    print(result["llm_answer"])

else:

    print("Error:",response.text)

RAG MOVIE RECOMMENDATION SYSTEM
Masukkan preferensi film : science fiction with high ratings

Top Retrieved Movies

1. Domain (2016)
2. The First (2014)
3. Alistair1918 (2016)
4. Story of Science, The (2010)
5. Scientist, The (2010)

Penjelasan LLM


You are movie recommendation assistant.

User Request:
science fiction with high ratings

Retrieved Movies:



Title : Domain (2016)

Genres : Drama|Sci-Fi|Thriller

Average Rating : 3.86



Title : The First (2014)

Genres : Sci-Fi

Average Rating : 2.50



Title : Alistair1918 (2016)

Genres : Sci-Fi

Average Rating : 2.50



Title : Story of Science, The (2010)

Genres : Documentary

Average Rating : 4.62



Title : Scientist, The (2010)

Genres : Drama

Average Rating : 3.62



Recommend the best movies with explanations.

